# Shared feature-selection development

Revised modeling run using one frozen, task-agnostic feature set. The original notebooks are preserved in `Archive/2026-10-08_before_feature_selection/`. All outputs from this run are written under `results/feature_selection_run/`.

# Updates
- 20260920 - V1: 
    - Initial testing of different models and operationability
    - "Make dataset" filters for delivered orders only - Not feasible for actual run because we won't know the actual status on any particular run date
- 20260926 - V2: 
    - Update "Make dataset" to align with actual operations
        - Only pull pXd training data and implement label completeness checks
    - Implement monthly inference backtesting
- 20261004 - V4:
    - Finalised notebook structure: Model dev pipeline -> CV + Hyperparam tuning -> Select best model
- 20201004 - V5:
    - Utilise helper functions to make dataset which includes extra features used in regression

# Environment Set-up

In [ ]:
# Install dependencies (uncomment if running in Colab)
# %pip install -q pandas numpy matplotlib seaborn plotly scikit-learn xgboost lightgbm

import json, os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))

import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
import xgboost as xgb
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score
)

from src.tuning import FoldWeightedPipeline
from src.preprocessing import make_preprocessor
from src import RANDOM_STATE, LOOKBACK, PXD
from src.data import load_olist_data
from src.features import (build_feature_table, add_extra_features,
                          NUM_COLS, CAT_COLS, EXTRA_NUM_COLS, LEAKAGE_COLS, SELECTED_NUM_COLS, SELECTED_CAT_COLS)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

# Constants

In [ ]:
NUM_COLS = SELECTED_NUM_COLS.copy()
CAT_COLS = SELECTED_CAT_COLS.copy()
FEATURE_COLS = NUM_COLS + CAT_COLS
print(f"Number of features: {len(FEATURE_COLS)}")
print(f"Number of numeric features: {len(NUM_COLS)}")
print(f"Number of categorical features: {len(CAT_COLS)}")
LABEL_COL = 'dpt_var_late_delivery'
TUNING_TEST_DAYS = 30  # Development holdout length


## Frozen feature method

Both tasks, every fold and every model family use one feature set frozen before modelling. The 41 approval-time predictors are reduced to **24 raw features** (21 numeric `SELECTED_NUM_COLS` plus 3 categorical `SELECTED_CAT_COLS`: `customer_state`, `route_type` and `payment_combination`), which reference coding expands to **53 encoded columns**. The reduction, its evidence and its limitations are reproduced in **Handling multicollinearity** below (summary in `MULTICOLLINEARITY_NOTES.md`).

# Load data and create features

The full approval-time master table is retained for auditing and target construction. Models use `SELECTED_NUM_COLS` and `SELECTED_CAT_COLS` only. Outcome columns are excluded from predictors. The frozen choices and overlap/VIF evidence are recorded in the shared feature audit.

In [ ]:
olist = load_olist_data()
orders = olist['orders']
final_df = add_extra_features(build_feature_table(olist), olist)
print(f'Shape: {final_df.shape}; unique orders: {final_df["order_id"].nunique():,}')

## Inspect data

In [ ]:
null_counts = final_df.isna().sum()
display(null_counts[null_counts > 0].sort_values(ascending=False))

print(f"Shape: {final_df.shape}")
print(f"Unique order_id: {final_df['order_id'].nunique()}")
print(final_df.shape)

display(final_df.head())

# Label definition

- This section is used to determine a suitable lookback buffer period at time of the model run date to ensure sufficient label availability based on the delivery outcomes that are known on the model run date itself.
- Here, we want to have at least 99% availability of the order labels

**Outcomes definitions:**

| Outcome as of model run date | Label |
|---|---|
| Delivered on or before the promised delivery day | **0 - On time** |
| Delivered after the promised delivery day | **1 - Late** |
| Still undelivered and the promised delivery day has fully passed | **1 - Late** |
| Still undelivered and the promised delivery day has not fully passed | **Unknown - Excluded from training** |

A delivery dated on or after the model run date is treated as not yet observed. 
- All comparisons use calendar dates
- Missing or invalid required dates leave the label unknown
- For simplicity, we assume that the promised delivery day remains on time until the following midnight


**Unknown labels have two subcategories:**

1. `unresolved`: the order is still undelivered through the day before the run date, and its estimated delivery day has not fully passed
2. `invalid_or_missing_dates`: required dates are missing or inconsistent, so a reliable label cannot be assigned. These orders are counted in `invalid_dates`

Both subcategories retain a missing `label_as_of_run`: **unknown_labels = unresolved orders + invalid_dates**.

For example, suppose the model runs on **June 2, 2018 (using data through June 1)**:

| Approval date | Estimated delivery date | Actual delivery date in the dataset | Subcategory | Why the label is unknown at the run time |
|---|---|---|---|---|
| May 2, 2018 | June 6, 2018 | June 4, 2018 | `unresolved` | Delivery is still in the future at the run time, and the deadline has not passed. |
| May 2, 2018 | June 2, 2018 | Missing | `unresolved` | The order still has the remainder of June 2 to arrive on time. |
| May 2, 2018 | Missing | May 21, 2018 | `invalid_or_missing_dates` | The order was delivered, but its estimated date is missing, so lateness cannot be determined. |
| May 6, 2018 | May 21, 2018 | May 4, 2018 | `invalid_or_missing_dates` | Delivery appears to precede approval, indicating inconsistent dates. |

In contrast, an order estimated for **June 1, 2018** that is still undelivered at this run time will be assigned with a **known label of 1 (late)**, assuming its required dates are valid because its promised delivery day has fully passed

## Check buffer lengths at each model training date
- Review label availability at historical run times
- Assumes delivery events are recorded immediately and available by end of the day at 2359H
- Unknown labels include invalid/missing dates which must be excluded from training

In [ ]:
BUFFER_RUN_DATES = ['2018-04-01', '2018-05-01', '2018-06-01']
BUFFER_DAYS = [15, 30, 45, 60]
BUFFER_WINDOW_DAYS = PXD
BUFFER_COMPLETENESS_TARGET = 99.0  # To set an acceptable threshold

label_orders = orders.loc[
        orders['order_id'].isin(final_df['order_id']),
        ['order_id', 'order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']
    ].drop_duplicates('order_id').copy()
for col in label_orders.columns.drop('order_id'):
    label_orders[col] = pd.to_datetime(label_orders[col])
label_orders['order_approved_dt'] = label_orders['order_approved_at'].dt.normalize()


def labels_as_of(order_rows, run_date: str):
    """Assign labels using calendar dates strictly before run_date (YYYY-MM-DD)."""

    if not isinstance(run_date, str):
        raise ValueError('run_date must be a YYYY-MM-DD string')
    run_day = pd.to_datetime(run_date, format='%Y-%m-%d', errors='coerce')
    
    result = order_rows.copy()

    # Obtain Series based on required dates
    approved = pd.to_datetime(result['order_approved_at']).dt.normalize()
    delivered = pd.to_datetime(result['order_delivered_customer_date']).dt.normalize()
    deadline = pd.to_datetime(result['order_estimated_delivery_date']).dt.normalize()

    # Define orders
    valid = approved.notna() & approved.lt(run_day) & deadline.notna()
    invalid_delivery = delivered.notna() & delivered.lt(approved) # Flag delivery dates before approval dates
    valid = valid & (~invalid_delivery) # Exclude invalid orders
    delivered_by_run = delivered.notna() & delivered.lt(run_day) # Returns a Boolean Series
    overdue = deadline.lt(run_day)

    result['delivered_by_run'] = delivered_by_run
    result['label_as_of_run'] = pd.Series(pd.NA, index=result.index, dtype='Int64')
    
    # Create label_status
    on_time = valid & delivered_by_run & delivered.le(deadline)
    late_delivered = valid & delivered_by_run & delivered.gt(deadline)
    late_undelivered = valid & ~delivered_by_run & overdue

    result['label_status'] = 'unresolved'
    result.loc[~valid, 'label_status'] = 'invalid_or_missing_dates'
    result.loc[on_time, 'label_status'] = 'on_time_delivered'
    result.loc[late_delivered, 'label_status'] = 'late_delivered'
    result.loc[late_undelivered, 'label_status'] = 'late_undelivered' # Run date already exceeded estimated delivery date and not delivered yet

    # Create labels
    result.loc[on_time, 'label_as_of_run'] = 0
    result.loc[late_delivered | late_undelivered, 'label_as_of_run'] = 1

    return result


buffer_rows = []
buffer_label_audits = {}  # Inspect an individual run/buffer's order-level labels.
for run_date in BUFFER_RUN_DATES:
    run_day = pd.Timestamp(run_date).normalize()
    inference_date = run_day - pd.Timedelta(days=1)
    as_of = labels_as_of(label_orders, run_date=run_date)
    for buffer_days in BUFFER_DAYS:
        end_date = inference_date - pd.Timedelta(days=buffer_days)
        start_date = end_date - pd.Timedelta(days=BUFFER_WINDOW_DAYS - 1)
        window = as_of.loc[as_of['order_approved_dt'].between(start_date, end_date)].copy()
        buffer_label_audits[(run_date, buffer_days)] = window
        for scope, scope_start in [
            ('full_window', start_date),
            ('newest_7_days', max(start_date, end_date - pd.Timedelta(days=6)))
        ]:
            cohort = window.loc[window['order_approved_dt'].ge(scope_start)]
            total = len(cohort)
            known = int(cohort['label_as_of_run'].notna().sum())
            completeness = 100 * known / total if total else float('nan')
            buffer_rows.append({
                'run_date': run_day, 
                'buffer_days': buffer_days,
                'scope': scope, 
                'start_date': scope_start, 
                'end_date': end_date,
                'orders': total, 
                'known_labels': known,
                'unknown_labels': total - known,
                'on_time_delivered': int(cohort['label_status'].eq('on_time_delivered').sum()),
                'late_delivered': int(cohort['label_status'].eq('late_delivered').sum()),
                'late_undelivered': int(cohort['label_status'].eq('late_undelivered').sum()),
                'total_late_delivery_rate_pct': 100 * cohort['label_as_of_run'].eq(1).sum() / known if known else float('nan'),
                'delivered_by_run_pct': 100 * cohort['delivered_by_run'].sum() / total if total else float('nan'),
                'label_completeness_pct': completeness,
                'meets_target': bool(total and completeness >= BUFFER_COMPLETENESS_TARGET)
            })
buffer_review = pd.DataFrame(buffer_rows)

for scope in ['full_window', 'newest_7_days']:
    print(f'\n{scope}:')
    display(buffer_review.loc[buffer_review['scope'].eq(scope)].round(2))

buffer_summary = buffer_review.groupby('buffer_days', as_index=False).agg(
    worst_completeness_pct=('label_completeness_pct', 'min'),
    all_runs_and_scopes_meet_target=('meets_target', 'all')
)
display(buffer_summary.round(2))

# Final evaluation of buffer period
passing_buffers = buffer_summary.loc[buffer_summary['all_runs_and_scopes_meet_target'], 'buffer_days']
if passing_buffers.empty:
    print(f'No candidate meets {BUFFER_COMPLETENESS_TARGET}% in every run and scope.')
else:
    print(f'Smallest candidate meeting {BUFFER_COMPLETENESS_TARGET}% in every run and scope: '
          f'{passing_buffers.min()} days. Note that this is only historical evidence and not a future guarantee.')

# Get training data
- To operationalise this pipeline, we need to run predictions on a daily basis, for example:
    - The prediction job is set to run at 0030H daily
    - Assuming at 2018-06-02 0030H, the orders approved on 2018-06-01 would have all came in. We want to predict whether the orders approved on 2018-06-01 will be late.
    - To do so, we use historical orders ending 45 days before the inference date, retaining labels known before the run date which is the lookback buffer period we have determined earlier to ensure label availability.
        - Run date: 2018-06-02
        - Inference date: 2018-06-01 (we are predicting for orders approved on this date)
        - June candidate history: 2017-04-18 to 2018-04-17 (365 days), ending 45 days before inference
        - The newest 30 days are the development holdout; a 45-day gap separates it from the training rows
    - pXd sets the total historical candidate window used for development (training, internal gap and holdout).

In [ ]:
def get_required_dates(run_date: str, lookback: int = 45, pXd: int = 180):
    """Return (start, end, inference_date) as YYYY-MM-DD strings; window is inclusive."""
    if isinstance(pXd, bool) or not isinstance(pXd, int) or pXd <= 0:
        raise ValueError('pXd must be a positive integer')
    if isinstance(lookback, bool) or not isinstance(lookback, int) or lookback < 0:
        raise ValueError('lookback must be a non-negative integer')

    if not isinstance(run_date, str):
        raise ValueError('run_date must be a YYYY-MM-DD string')
    run_day = pd.to_datetime(run_date, format='%Y-%m-%d', errors='coerce')
    if pd.isna(run_day) or run_day.strftime('%Y-%m-%d') != run_date:
        raise ValueError('run_date must be a valid date in YYYY-MM-DD format')
    inference_date = run_day - pd.Timedelta(days=1)

    test_end_date = inference_date - pd.Timedelta(days=lookback)
    test_start_date = test_end_date - pd.Timedelta(days=TUNING_TEST_DAYS-1)
    train_end_date = test_start_date - pd.Timedelta(days=1)
    train_start_date = train_end_date - pd.Timedelta(days=pXd-TUNING_TEST_DAYS-1)

    inference_date_str = inference_date.strftime('%Y-%m-%d')
    test_end_date = test_end_date.strftime('%Y-%m-%d')
    test_start_date = test_start_date.strftime('%Y-%m-%d')
    train_end_date = train_end_date.strftime('%Y-%m-%d')
    train_start_date = train_start_date.strftime('%Y-%m-%d')

    print(f"Train-test period for the inference date of {inference_date_str} with buffer of {lookback} days and pXd of {pXd} days:")
    print(f"   Candidate window before applying the holdout gap: {train_start_date} to {train_end_date} "
          f"({pXd-TUNING_TEST_DAYS} calendar days)")
    print(f"   Development holdout: {test_start_date} to {test_end_date} "
          f"({TUNING_TEST_DAYS} calendar days)")

    return train_start_date, train_end_date, test_start_date, test_end_date, inference_date_str

run_date = '2018-06-02'
train_start_date, train_end_date, test_start_date, test_end_date, inference_date = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD)

In [ ]:
train_test_df = final_df[
    final_df['order_approved_dt'].between(
        train_start_date,
        test_end_date,
        inclusive='both'
    )
].copy()
train_test_df['order_approved_dt'].value_counts().sort_index()

## Check label availability
- This section serves as a label availability check during each training run in production
- It does not filter rows or replace existing targets yet
- Use label_as_of_run from the audit for binary training labels at this run timestamp

In [ ]:
# Use dates before run_date; the operational 00:30 schedule has no effect.
LABEL_AVAILABILITY_TARGET_PCT = 99.0

# Count each order once and retain the order-level audit for inspection.
## In train_test_df, the last approval date is 45 days before the run_date
train_test_label_audit = labels_as_of(train_test_df.drop_duplicates('order_id'), run_date=run_date)

total_orders = len(train_test_label_audit)
known_labels = int(train_test_label_audit['label_as_of_run'].notna().sum())
unknown_labels = total_orders - known_labels
unresolved_labels = int(train_test_label_audit['label_status'].eq('unresolved').sum())
invalid_dates = int(train_test_label_audit['label_status'].eq('invalid_or_missing_dates').sum())
label_availability_pct = 100 * known_labels / total_orders if total_orders else float('nan')

print(f'Model run on {run_date}:')
print(f'Historical candidate approvals: {train_start_date} to {test_end_date} '
      f'({PXD} days; {LOOKBACK}-day inference buffer)')
if total_orders:
    print(f'Labels available: {known_labels:,}/{total_orders:,} ({label_availability_pct:.2f}%)')
    print(f'Unknown labels: {unknown_labels:,} '
          f'(unresolved: {unresolved_labels:,}; invalid/missing dates: {invalid_dates:,})')
    if label_availability_pct >= LABEL_AVAILABILITY_TARGET_PCT:
        print(f'PASS: label availability meets the {LABEL_AVAILABILITY_TARGET_PCT:g}% target.')
    else:
        print(f'WARNING: label availability is below the {LABEL_AVAILABILITY_TARGET_PCT:g}% target.')
    if unknown_labels:
        print('Exclude unknown labels from supervised training, even when the availability target is met.')
else:
    print('WARNING: no orders in the selected training/test period; label availability cannot be calculated.')

## Determine late delivery labels
- Refer to label definition above
- Unknown outcomes are excluded

In [ ]:
# Retain only orders with a known label at the model run date.
train_test_label = train_test_label_audit.loc[
    train_test_label_audit['label_as_of_run'].notna()
].copy()

# Use the audited binary target, including overdue undelivered orders.
train_test_label[LABEL_COL] = train_test_label['label_as_of_run'].astype(int)

print(f'Removed {len(train_test_label_audit) - len(train_test_label):,} orders with unknown labels.')
print(f'Retained {len(train_test_label):,} orders with known labels.')
display(train_test_label['label_status'].value_counts())
display(train_test_label[LABEL_COL].value_counts(normalize=True))

display(train_test_label.head())

## Development holdout split
- The newest 30 days form the development holdout for model selection.
- Keep a 45-day gap before the holdout and retain only training labels known at its start.
- Default and tuned candidates use the same training and holdout rows.

In [ ]:
from src.splits import chronological_split, available_training_rows

# Split the 365-day candidate window, leaving a 45-day gap before the holdout.
# Default and tuned candidates use these same training and holdout rows.
window_df = train_test_label[
    train_test_label['order_approved_dt'].between(train_start_date, test_end_date, inclusive='both')
].copy()
train_df, test_df = chronological_split(
    window_df, date_col='order_approved_dt', test_days=TUNING_TEST_DAYS, gap_days=LOOKBACK)
train_df = available_training_rows(
    train_df, test_df['order_approved_dt'].min(), LABEL_COL,
    lambda rows, date: labels_as_of(rows, run_date=date), 'label_as_of_run')

train_df = train_df.sort_values('order_approved_dt', kind='stable')
X_train, y_train = train_df[FEATURE_COLS], train_df[LABEL_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[LABEL_COL]

print(f"\nData split complete:")
print(f"   Training: {len(X_train):,} samples ({y_train.mean()*100:.2f}% late)")
print(f"   Holdout:  {len(X_test):,} samples ({y_test.mean()*100:.2f}% late)")
print(f"   Gap before holdout: {LOOKBACK} days; training labels checked as of "
      f"{test_df['order_approved_dt'].min():%Y-%m-%d}")

# Handling multicollinearity

**Why it matters here.** In Linear/Ridge regression and logistic regression, strongly correlated predictors make coefficients unstable (large variance, sign changes between folds) and impossible to read as the effect of one predictor with the others held fixed; an exact identity makes the design singular. Tree models still predict well, but split importance arbitrarily between near-duplicates, so importance rankings understate each of them. Because one feature set is shared by both tasks and all model families, redundancy is removed once, before modelling, using only the June development-training rows of both tasks (45,972 classification and 45,371 regression orders). No holdout outcome or model score is used.

**Funnel (41 → 24 raw features).** Numeric VIF uses each task's median-imputed matrix with an intercept; the iterative stage drops the column with the greatest VIF in either task until every VIF ≤ 5 in both.

| Stage | Removed | Numeric remaining | Columns removed and rule |
|---|---:|---:|---|
| Start | – | 39 (+ 2 categorical = 41) | `NUM_COLS` (29) + `EXTRA_NUM_COLS` (10); `customer_state`, `route_type` |
| Constants | 2 | 37 | `payment_type_value_not_defined`, `payment_type_count_not_defined` |
| Domain representative | 13 | 24 | `order_approved_week_of_year` (month and day cover timing); `order_revenue_sum` (exactly price + freight); `seller_dist_km_min/max/median` (equal to the mean for ~99% of orders); 8 per-method payment value/count columns (replaced by `payment_value_sum` + categorical `payment_combination`) |
| Iterative joint VIF ≤ 5 | 3 | 21 | `order_pdt_price_sum` (VIF 65.9 / 23,819), `seller_zip_code_prefix_count` (18.1), `seller_city_count` (5.9) |
| **Final** | **18** | **21** | + 3 categorical (`payment_combination` added) = **24 raw features**; max numeric VIF 3.76 (classification) / 3.79 (regression) |

**Decisions.**
1. *Payment total over product price.* `payment_value_sum` and `order_pdt_price_sum` are near-duplicates (Spearman 0.99). Payment total is present for every order, including 412 classification training orders with no item rows (price and revenue missing), so it is exempt from the VIF stage, which then removes price.
2. *Distance and route both kept.* In the encoded design `seller_dist_km_mean` reaches VIF 6.04 (classification) / 6.22 (regression) because it overlaps `route_type`: same-state orders have median distance 100 km versus 663 km for interstate. Distance measures how far; route measures whether the shipment crosses a state border. This is the single documented exception to VIF ≤ 5.
3. *Nonlinear calendar flags kept.* `is_weekend_approval`, `is_black_friday_period` and `is_december_peak` are deterministic functions of weekday, month and day, but nonlinear ones, so VIF (a linear diagnostic) cannot see the dependence (VIF 2.16–2.36). They are kept deliberately to give the linear models step effects that trees can find for themselves.

**Encoding (24 raw → 53 columns).** Every row has exactly one level of each categorical, so a full set of dummies sums to the intercept column: with an intercept, full one-hot coding is perfectly collinear (the dummy-variable trap) and coefficients are not unique. `make_preprocessor(..., reference_categories=True)` therefore drops one reference level per categorical, and each dummy coefficient is a contrast with that reference.

| Block | Levels seen | Reference dropped | Encoded columns |
|---|---:|---|---:|
| Numeric | – | – | 21 |
| `customer_state` | 27 | `SP` | 26 |
| `route_type` | 3 | `1. All interstate` | 2 |
| `payment_combination` | 5 | `credit_card` | 4 |
| **Total** | | | **53** (55 without reference coding) |

The 53-column design is full rank after centring in both tasks; the largest encoded VIF is the distance/route overlap above.

**Limitations.** Selection uses only the June development-training rows and is applied unchanged to every fold and month, so earlier historical folds use a choice informed by later data. VIF detects linear dependence only. Category dummy coefficients are relative to their reference level, not absolute effects. Encoder categories are learned per training fold, so earlier folds can have fewer than 53 columns; categories unseen in training encode identically to the reference.


In [ ]:
from pathlib import Path
from src.feature_selection import (audit, encoded_block_summary, exact_overlap_audit,
                                   imputed_numeric, numeric_vif, stage_summary, training_rows)

MC_TASK = 'classification'
MC_DIR = Path('results/feature_selection_run/multicollinearity')
mc_rows = training_rows(final_df)  # June development-training rows for both tasks
mc_ids, nb_ids = set(mc_rows[MC_TASK]['order_id']), set(train_df['order_id'])
print(f'audit rows: {len(mc_ids):,}; notebook train_df: {len(nb_ids):,}; '
      f'shared: {len(mc_ids & nb_ids):,}; audit only: {len(mc_ids - nb_ids):,}; '
      f'notebook only: {len(nb_ids - mc_ids):,}')
assert mc_ids == nb_ids, 'notebook training rows differ from the audit population'

In [ ]:
# Exact identities: revenue = price + freight; payment total = sum of method values;
# share of orders whose seller distance max/min equals the mean.
display(pd.DataFrame(exact_overlap_audit(mc_rows[MC_TASK])))

In [ ]:
# Joint audit over both tasks; writes the before/after VIF, trace and encoded-design CSVs.
mc_selected = audit({task: imputed_numeric(rows) for task, rows in mc_rows.items()}, mc_rows, output=MC_DIR)
assert mc_selected == NUM_COLS, 'audit result differs from the frozen numeric list'
mc_trace = pd.read_csv(MC_DIR / 'removal_trace.csv')
mc_before = pd.read_csv(MC_DIR / 'numeric_vif_before.csv', index_col='feature')
display(stage_summary(mc_trace, start_count=len(mc_before)))
display(mc_trace)

In [ ]:
mc_after = pd.read_csv(MC_DIR / 'numeric_vif_after.csv', index_col='feature')
display(mc_before[MC_TASK].sort_values(ascending=False).head(15).to_frame('VIF before (top 15)'))
display(mc_after[MC_TASK].sort_values(ascending=False).to_frame(f'VIF after ({len(mc_after)} retained)'))

In [ ]:
# Reference-coded design fitted on this notebook's training rows.
mc_preprocessor = make_preprocessor(NUM_COLS, CAT_COLS, reference_categories=True).fit(X_train)
display(encoded_block_summary(mc_preprocessor))
mc_design = pd.DataFrame(mc_preprocessor.transform(X_train), columns=mc_preprocessor.get_feature_names_out())
mc_rank = np.linalg.matrix_rank((mc_design - mc_design.mean()).to_numpy(), tol=1e-8)
print(f'Encoded columns: {mc_design.shape[1]}; rank of centred matrix: {mc_rank}')
display(numeric_vif(mc_design).sort_values(ascending=False).head(5).to_frame('encoded VIF (top 5)'))

# Preprocessor
- Missing numeric values are handled inside each model pipeline using training medians for Logistic Regression, Decision Tree, XGBoost, Random Forest and LightGBM.

In [ ]:
# Fit numeric medians and category mappings on training data inside each pipeline.
preprocessor = make_preprocessor(NUM_COLS, CAT_COLS, reference_categories=True)

# Model Training
- Instantiate the model pipelines and train the models

## Logistic Regression

In [ ]:
# Baseline with class balancing; preprocessing is fitted on training data only.
lr_preprocessor = make_preprocessor(NUM_COLS, CAT_COLS, scale=True, reference_categories=True)
lr_pipeline = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42,
    )),
])
lr_pipeline.fit(X_train, y_train)
print('Logistic Regression Pipeline trained!')

## Decision Tree

In [ ]:
# Limit tree depth and balance classes for the late-delivery target.
dt_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', DecisionTreeClassifier(
        max_depth=10,
        class_weight='balanced',
        random_state=42,
    )),
])
dt_pipeline.fit(X_train, y_train)
print('Decision Tree Pipeline trained!')

## Random Forest

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=100,           # Number of trees in the forest
    max_depth=10,               # Maximum tree depth
    class_weight='balanced',    # Auto-adjust weights inversely to class freq
    random_state=42,
    n_jobs=-1                   # Use all CPU cores for parallel training
)

# Create Pipeline with a fresh preprocessor
# (Each pipeline needs its own preprocessor to avoid conflicts)
rf_pipeline = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', rf_classifier)
])

# Fit the entire pipeline
rf_pipeline.fit(X_train, y_train)
print("   Random Forest Pipeline trained!")
print("   Pipeline steps: preprocessor -> classifier")

## XGBoost

In [ ]:
xgb_classifier = xgb.XGBClassifier(
    n_estimators=100,           # Number of boosting rounds (trees)
    max_depth=6,                # Maximum tree depth (controls complexity)
    learning_rate=0.1,          # Step size shrinkage (prevents overfitting)

    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Create Pipeline: preprocessor -> model
xgb_pipeline = FoldWeightedPipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_classifier)
])

# Fit the entire pipeline (preprocessor + model) on training data
xgb_pipeline.fit(X_train, y_train)
print("   XGBoost Pipeline trained!")
print("   Pipeline steps: preprocessor -> classifier")

## LightGBM

In [ ]:
lgb_pipeline = FoldWeightedPipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', LGBMClassifier(

        random_state=42, 
        verbosity=-1,
        n_jobs=-1,
    )),
])
lgb_pipeline.fit(X_train, y_train)
print('LightGBM Pipeline trained!')

# Evaluation - Test Set

In [ ]:
def evaluate_pipeline(pipeline, X_test, y_test):
    """
    Evaluate a pipeline and return metrics.

    Note: X_test contains RAW features (not preprocessed).
    The pipeline handles preprocessing internally.
    """
    # Pipeline handles preprocessing + prediction in one call
    y_pred = pipeline.predict(X_test)              # Binary: 0 or 1
    y_prob = pipeline.predict_proba(X_test)[:, 1]  # Probability of late delivery

    metrics = {
        'roc_auc': (roc_auc_score(y_test, y_prob)
                    if pd.Series(y_test).nunique() == 2 else np.nan),
        'avg_precision': (average_precision_score(y_test, y_prob)
                    if pd.Series(y_test).nunique() == 2 else np.nan),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        }

    return metrics, y_pred, y_prob

# Evaluate pipelines (note: X_test is RAW data, pipeline handles preprocessing)
lr_metrics, lr_pred, lr_prob = evaluate_pipeline(lr_pipeline, X_test, y_test)
dt_metrics, dt_pred, dt_prob = evaluate_pipeline(dt_pipeline, X_test, y_test)
rf_metrics, rf_pred, rf_prob = evaluate_pipeline(rf_pipeline, X_test, y_test)
xgb_metrics, xgb_pred, xgb_prob = evaluate_pipeline(xgb_pipeline, X_test, y_test)
lgb_metrics, lgb_pred, lgb_prob = evaluate_pipeline(lgb_pipeline, X_test, y_test)

print(f"Baseline precision: {y_test.mean():.2%}")

metrics_summary = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Decision Tree': dt_metrics,
    'Random Forest': rf_metrics,
    'XGBoost': xgb_metrics,
    'LightGBM': lgb_metrics,
}).T
display((metrics_summary * 100).style.format("{:.2f}%"))

model_scores = {
    'Logistic Regression': lr_prob,
    'Decision Tree': dt_prob,
    'Random Forest': rf_prob,
    'XGBoost': xgb_prob,
    'LightGBM': lgb_prob,
}

# Feature Importances

In [ ]:
# Logistic Regression: numeric features are standardized; categorical features
# remain 0/1 indicators. Coefficients describe changes in late-delivery log-odds.
lr_coefficients = lr_pipeline.named_steps['classifier'].coef_[0]
lr_top10 = (
    pd.DataFrame({
        'feature': lr_pipeline.named_steps['preprocessor'].get_feature_names_out(),
        'coefficient': lr_coefficients,
        'absolute_coefficient': np.abs(lr_coefficients),
    })
    .sort_values('absolute_coefficient', ascending=False)
    .head(10).reset_index(drop=True)
)


# Decision Tree: normalized total reduction in impurity per feature.
dt_top10 = (
    pd.DataFrame({
        'feature': dt_pipeline.named_steps['preprocessor'].get_feature_names_out(),
        'impurity_importance': dt_pipeline.named_steps['classifier'].feature_importances_
    })
    .sort_values('impurity_importance', ascending=False)
    .head(10).reset_index(drop=True)
)


rf_top10 = (
    pd.DataFrame({
        'feature': rf_pipeline.named_steps['preprocessor'].get_feature_names_out(),
        'impurity_importance': rf_pipeline.named_steps['classifier'].feature_importances_
    })
    .sort_values('impurity_importance', ascending=False)
    .head(10).reset_index(drop=True)
)


xgb_feature_names = xgb_pipeline.named_steps['preprocessor'].get_feature_names_out()
xgb_booster = xgb_pipeline.named_steps['classifier'].get_booster()
xgb_gain = xgb_booster.get_score(importance_type='gain')
xgb_feature_keys = xgb_booster.feature_names or [f'f{i}' for i in range(len(xgb_feature_names))]

xgb_top10 = (
    pd.DataFrame({
        'feature': xgb_feature_names,
        'gain': [xgb_gain.get(key, 0.0) for key in xgb_feature_keys]
    })
    .sort_values('gain', ascending=False)
    .head(10).reset_index(drop=True)
)


lgb_top10 = (
    pd.DataFrame({
        'feature': lgb_pipeline.named_steps['preprocessor'].get_feature_names_out(),
        'total_gain': lgb_pipeline.named_steps['classifier'].booster_.feature_importance(
            importance_type='gain'
        ),
    })
    .sort_values('total_gain', ascending=False)
    .head(10).reset_index(drop=True)
)


print('Logistic Regression: top 10 by absolute coefficient (positive = higher late-delivery odds)')
display(lr_top10)
print('Decision Tree: top 10 features by decrease in impurity')
display(dt_top10)
print('XGBoost: top 10 features by average gain')
display(xgb_top10)
print('Random Forest: top 10 features by mean decrease in impurity')
display(rf_top10)
print('LightGBM: top 10 by total split gain')
display(lgb_top10)

# Probability Distribution

In [ ]:
# Use the same bins and axis scales to compare all five models.
fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True, sharey=True)
bins = np.linspace(0, 1, 101)
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:purple', 'tab:brown']

for ax, (name, scores), color in zip(axes.flat, model_scores.items(), colors):
    ax.hist(scores, bins=bins, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(0.5, color='red', linestyle=':', linewidth=2, label='Score = 0.5')
    ax.set_title(name)
    ax.set_xlabel('Predicted late-delivery score')
    ax.set_ylabel('Number of orders')
    ax.set_xlim(0, 1)
    ax.tick_params(labelbottom=True)
    ax.legend()

for ax in list(axes.flat)[len(model_scores):]:
    ax.set_visible(False)

fig.suptitle('Test-set probability distributions')
plt.tight_layout()
plt.show()

# Backtesting - Inference Set
- Using the trained models, peform inference on a "live" (unseen) cohort of newly approved orders
- Orders eligibility:
    - These orders are not delivered yet, they are all eligible orders approved on the inference date
- Operation:
    - For June development, fit on June 2 historical data and score daily June cohorts
    - Each fitted candidate is used to predict all eligible days in June; the later experiment keeps the June-selected fitted pipeline unchanged through July and August
    - The predictions on these 30/31 days will then be collated to have sufficient data to compute the evaluation metrics and coverage on a monthly basis

**Monthly evaluation and label availability**

- Pool all daily predictions for the month after every cohort has reached its own cutoff. The outcome data must cover those cutoffs.
- Compute monthly metrics using known actual labels; report unknown outcomes separately.
- The historical buffer audit supports the 45-day choice against the **99% label-availability target**. Check and report availability for the inference cohort as well.
- Training uses completed days strictly before its shared model run date. Retrospective inference evaluation includes its entire per-cohort cutoff date. Both use the same date-based label rules.

In [ ]:
def get_month_start_end(run_date: str):
    """Return the first and last dates of run_date's month as YYYY-MM-DD strings."""
    from calendar import monthrange
    from datetime import date

    if not isinstance(run_date, str):
        raise ValueError('run_date must be a YYYY-MM-DD string')
    try:
        run_day = date.fromisoformat(run_date)
    except ValueError as exc:
        raise ValueError('run_date must be a valid date in YYYY-MM-DD format') from exc
    if run_day.isoformat() != run_date:
        raise ValueError('run_date must be a valid date in YYYY-MM-DD format')

    month_start = run_day.replace(day=1)
    month_end = run_day.replace(day=monthrange(run_day.year, run_day.month)[1])
    return month_start.isoformat(), month_end.isoformat()

inference_month_start_date, inference_month_end_date = get_month_start_end(run_date)
print(f'Days to run inference: {inference_month_start_date} to {inference_month_end_date}')

## Perform inference loop
- Loop over the inference days to collect all the predictions and perform evaluation

In [ ]:
def run_inference(inference_month_start_date, inference_month_end_date, model_pipeline):
    """Score eligible orders over an inclusive date range using a fitted pipeline.

    Uses the notebook's final_df and FEATURE_COLS. Orders must be approved on
    the inference day and not delivered by that day. Preprocessing is not refitted.
    Returns approval date, order ID and late-delivery probability for each order.
    """
    start_day = pd.to_datetime(inference_month_start_date).normalize()
    end_day = pd.to_datetime(inference_month_end_date).normalize()
    if pd.isna(start_day) or pd.isna(end_day) or start_day > end_day:
        raise ValueError('Inference dates must be valid and start must not exceed end.')

    daily_prediction_frames = []
    late_class_index = list(model_pipeline.classes_).index(1)
    approval_days = pd.to_datetime(final_df['order_approved_dt']).dt.normalize()
    delivery_days = pd.to_datetime(final_df['order_delivered_customer_date']).dt.normalize()

    for inference_day in pd.date_range(start_day, end_day, freq='D'):
        # Only deliveries through the inference day are already observed.
        eligible = approval_days.eq(inference_day) & (
            delivery_days.isna() | delivery_days.gt(inference_day)
        )
        daily_orders = final_df.loc[eligible]
        if daily_orders.empty:
            continue

        daily_features = daily_orders[FEATURE_COLS].copy()
        daily_predictions = daily_orders[['order_approved_dt', 'order_id']].copy()
        daily_predictions['order_approved_dt'] = inference_day
        daily_predictions['predicted_probability'] = model_pipeline.predict_proba(
            daily_features
        )[:, late_class_index]
        daily_prediction_frames.append(daily_predictions)

    inference_predictions = (
        pd.concat(daily_prediction_frames, ignore_index=True)
        .sort_values(['order_approved_dt', 'order_id'], ignore_index=True)
        if daily_prediction_frames else pd.DataFrame({
            'order_approved_dt': pd.Series(dtype='datetime64[ns]'),
            'order_id': pd.Series(dtype='object'),
            'predicted_probability': pd.Series(dtype='float64'),
        })
    )
    model_name = type(model_pipeline.named_steps['classifier']).__name__
    print(f'{model_name} predictions: \n{len(inference_predictions):,} orders between '
          f'{start_day:%Y-%m-%d} and {end_day:%Y-%m-%d} (inclusive), across '
          f'{inference_predictions["order_approved_dt"].nunique()} days with eligible orders\n')
    return inference_predictions


lr_inference_predictions = run_inference(inference_month_start_date, inference_month_end_date, lr_pipeline)
dt_inference_predictions = run_inference(inference_month_start_date, inference_month_end_date, dt_pipeline)
rf_inference_predictions = run_inference(inference_month_start_date, inference_month_end_date, rf_pipeline)
xgb_inference_predictions = run_inference(inference_month_start_date, inference_month_end_date, xgb_pipeline)
lgb_inference_predictions = run_inference(inference_month_start_date, inference_month_end_date, lgb_pipeline)

# Sample model: inspect predicted probabilities for LGBM model
display(lgb_inference_predictions)

### Attach actual and predicted labels

**Target and predicted labels**

- Target: **failed to arrive by the promised delivery date**.
- Predicted label **1 (late)**: model score is greater than or equal to `PREDICTION_THRESHOLD` (0.5).
- Predicted label **0 (on time)**: model score is below 0.5.
- Daily predictions use the same fitted monthly model; attaching actual labels later does not change the saved scores

In [ ]:
PREDICTION_THRESHOLD = 0.5

def attach_prediction_labels(predictions, lookback=LOOKBACK,
                             prediction_threshold=PREDICTION_THRESHOLD, *,
                             model_name='Model'):
    """Return a labelled copy of predictions using inclusive approval-day cutoffs.

    Requires order_id and predicted_probability columns. Uses the notebook's
    orders table and labels_as_of helper. Reports the count and percentage of
    known actual labels at approval day + lookback for the supplied model.
    The prediction threshold defaults to PREDICTION_THRESHOLD (0.5).
    """
    if isinstance(lookback, bool) or not isinstance(lookback, int) or lookback < 0:
        raise ValueError('lookback must be a non-negative integer')
    labelled_predictions = predictions.copy()
    # Inclusive cutoff date: observe the whole approval day + lookback days.
    # Reuse the training label rules, with a separate cutoff for each daily cohort.
    outcomes = orders[['order_id', 'order_approved_at', 'order_delivered_customer_date',
                       'order_estimated_delivery_date']].drop_duplicates('order_id').copy()
    outcomes = outcomes.loc[outcomes['order_id'].isin(labelled_predictions['order_id'])].copy()

    for column in ['order_approved_at', 'order_delivered_customer_date',
                   'order_estimated_delivery_date']:
        outcomes[column] = pd.to_datetime(outcomes[column])
    
    outcomes['evaluation_cutoff'] = (outcomes['order_approved_at'].dt.normalize() + pd.Timedelta(days=lookback))
    outcomes['actual_label'] = pd.Series(pd.NA, index=outcomes.index, dtype='Int64')
    outcomes['label_status'] = 'invalid_or_missing_dates'
    for cutoff, cohort in outcomes.groupby('evaluation_cutoff'):
        # The helper excludes its run date; use the next day to include the full cutoff day.
        audit_day = cutoff + pd.Timedelta(days=1)
        audit = labels_as_of(cohort, run_date=audit_day.strftime('%Y-%m-%d'))
        outcomes.loc[audit.index, 'actual_label'] = audit['label_as_of_run']
        outcomes.loc[audit.index, 'label_status'] = audit['label_status']

    # Map by order ID so rerunning does not duplicate rows or columns.
    outcome_lookup = outcomes.set_index('order_id')
    for column in ['actual_label', 'label_status', 'evaluation_cutoff']:
        labelled_predictions[column] = (
            labelled_predictions['order_id'].map(outcome_lookup[column])
        )
    labelled_predictions['actual_label'] = (labelled_predictions['actual_label'].astype('Int64'))
    labelled_predictions['predicted_label'] = (labelled_predictions['predicted_probability'] >= prediction_threshold).astype(int)

    known_labels = int(labelled_predictions['actual_label'].notna().sum())
    total_predictions = len(labelled_predictions)
    label_completeness_pct = ((known_labels / total_predictions * 100) if total_predictions else np.nan)
    availability_pct = f'{label_completeness_pct:.2f}%' if total_predictions else 'N/A'
    print(f'{model_name}: known actual labels at approval day + {lookback} days: '
          f'{known_labels:,}/{total_predictions:,} ({availability_pct})')
    if total_predictions:
        availability_result = (
            'PASS' if label_completeness_pct >= BUFFER_COMPLETENESS_TARGET else 'WARNING'
        )
        print(f'{availability_result}: inference label availability versus '
              f'{BUFFER_COMPLETENESS_TARGET:g}% historical target.')

    return labelled_predictions


lr_inference_predictions = attach_prediction_labels(lr_inference_predictions, model_name='Logistic Regression')
dt_inference_predictions = attach_prediction_labels(dt_inference_predictions, model_name='Decision Tree')
rf_inference_predictions = attach_prediction_labels(rf_inference_predictions, model_name='Random Forest')
xgb_inference_predictions = attach_prediction_labels(xgb_inference_predictions, model_name='XGBoost')
lgb_inference_predictions = attach_prediction_labels(lgb_inference_predictions, model_name='LightGBM')

# Sample model: inspect label-status counts and the labelled prediction table.
display(lgb_inference_predictions['label_status'].value_counts(dropna=False))
display(lgb_inference_predictions)

### Evaluate monthly predictions

Calculate accuracy, precision, recall and F1 for orders with known actual labels, treating late delivery (1) as positive. Unknown outcomes are excluded, so metrics describe the observed-outcome subset. Undefined precision/recall/F1 return zero; an empty evaluation set returns missing metrics.

In [ ]:
def evaluate_monthly_predictions(predictions, model_name):
    """Return one row of metrics for known actual labels, indexed by model name.

    Requires actual_label, predicted_label and predicted_probability from attach_prediction_labels().
    Late delivery (1) is positive. Unknown actual labels are excluded;
    an empty evaluation set returns missing metrics.
    """
    evaluation_rows = predictions.dropna(subset=['actual_label'])
    if evaluation_rows.empty:
        metric_values = dict.fromkeys(['roc_auc', 'avg_precision', 'accuracy', 'precision', 'recall', 'f1_score'], np.nan)
    else:
        actual = evaluation_rows['actual_label'].astype(int)
        predicted = evaluation_rows['predicted_label']
        probability = evaluation_rows['predicted_probability']
        two_classes = actual.nunique() == 2
        metric_values = {
            'roc_auc': roc_auc_score(actual, probability) if two_classes else np.nan,
            'avg_precision': average_precision_score(actual, probability) if two_classes else np.nan,
            'accuracy': accuracy_score(actual, predicted),
            'precision': precision_score(actual, predicted, zero_division=0),
            'recall': recall_score(actual, predicted, zero_division=0),
            'f1_score': f1_score(actual, predicted, zero_division=0),
            }
    print(f'{model_name}: evaluated {len(evaluation_rows):,} orders; '
          f'excluded {len(predictions) - len(evaluation_rows):,} unknown outcomes.')
    return pd.DataFrame([metric_values], index=pd.Index([model_name], name='model'))


lr_inference_metrics = evaluate_monthly_predictions(lr_inference_predictions, 'Logistic Regression')
dt_inference_metrics = evaluate_monthly_predictions(dt_inference_predictions, 'Decision Tree')
rf_inference_metrics = evaluate_monthly_predictions(rf_inference_predictions, 'Random Forest')
xgb_inference_metrics = evaluate_monthly_predictions(xgb_inference_predictions, 'XGBoost')
lgb_inference_metrics = evaluate_monthly_predictions(lgb_inference_predictions, 'LightGBM')

monthly_results_df = pd.concat([
        lr_inference_metrics,
        dt_inference_metrics,
        rf_inference_metrics,
        xgb_inference_metrics,
        lgb_inference_metrics,
        ])
display(monthly_results_df.style.format('{:.2%}', na_rep='N/A'))

### Coverage by ranked probability deciles
To assess the performance of the model, we should determine whether:
- It beats existing targeting methods
- Captures enough actionable late orders within capacity
- Produces positive value consistently across future months

In [ ]:
def display_decile_coverage(predictions, model_name):
    """Display decile coverage and return (coverage, ranked_predictions).

    Requires order_id, predicted_probability and actual_label. Rank all scored
    orders into nearly equal-sized rank groups, with highest scores in decile 1.
    Equal scores are split using ascending order_id as the tie-breaker. With
    fewer than ten orders, use one group per order. Unknown actual labels count
    as orders but are excluded from late-delivery rates and coverage denominators.
    Inputs are copied so the model's original predictions remain unchanged.
    """
    ranked_predictions = predictions.sort_values(
        ['predicted_probability', 'order_id'], ascending=[False, True], kind='stable'
    ).reset_index(drop=True).copy()
    row_count = len(ranked_predictions)
    ranked_predictions['probability_rank'] = np.arange(1, row_count + 1)

    # Rank-based groups differ in size by at most one order, even with tied scores.
    # Handle empty/single-order cohorts explicitly because qcut needs distinct ranks.
    ranked_predictions['decile'] = (
        pd.Series(1, index=ranked_predictions.index, dtype='Int64')
        if row_count <= 1
        else (pd.qcut(
            ranked_predictions['probability_rank'],
            q=min(10, row_count),
            labels=False,
        ) + 1).astype('Int64')
    )
    coverage = (
        ranked_predictions.groupby('decile').agg(
            number_of_orders=('order_id', 'size'),
            known_labels=('actual_label', 'count'),
            positive_labels=('actual_label', 'sum')
        ).reindex(range(1, 11), fill_value=0).rename_axis('decile').reset_index()
    )
    coverage['unknown_labels'] = coverage['number_of_orders'] - coverage['known_labels']
    coverage['cumulative_orders'] = coverage['number_of_orders'].cumsum()
    coverage['cumulative_known_labels'] = coverage['known_labels'].cumsum()
    coverage['cumulative_positive_labels'] = coverage['positive_labels'].cumsum()
    coverage['late_delivery_rate'] = (
        coverage['positive_labels'] / coverage['known_labels'].replace(0, np.nan)
    )
    coverage['cumulative_late_delivery_rate'] = coverage['cumulative_positive_labels'] / coverage['cumulative_known_labels'].replace(0, np.nan)
    total_positive_labels = coverage['positive_labels'].sum()
    coverage['coverage'] = (
        coverage['cumulative_positive_labels'] / total_positive_labels
        if total_positive_labels > 0 else np.nan
    )

    print(f'{model_name}: monthly decile coverage')
    display(coverage.style.format({
                'late_delivery_rate': '{:.2%}',
                'cumulative_late_delivery_rate': '{:.2%}',
                'coverage': '{:.2%}'
                }, na_rep='N/A')
            )

    return coverage, ranked_predictions

lr_inference_coverage, lr_ranked_predictions = display_decile_coverage(lr_inference_predictions, 'Logistic Regression')
dt_inference_coverage, dt_ranked_predictions = display_decile_coverage(dt_inference_predictions, 'Decision Tree')
rf_inference_coverage, rf_ranked_predictions = display_decile_coverage(rf_inference_predictions, 'Random Forest')
xgb_inference_coverage, xgb_ranked_predictions = display_decile_coverage(xgb_inference_predictions, 'XGBoost')
lgb_inference_coverage, lgb_ranked_predictions = display_decile_coverage(lgb_inference_predictions, 'LightGBM')

# Compare cumulative late-order coverage across all five models.
decile_coverage_comparison = pd.concat({
    'Logistic Regression': lr_inference_coverage.set_index('decile')['coverage'],
    'Decision Tree': dt_inference_coverage.set_index('decile')['coverage'],
    'Random Forest': rf_inference_coverage.set_index('decile')['coverage'],
    'XGBoost': xgb_inference_coverage.set_index('decile')['coverage'],
    'LightGBM': lgb_inference_coverage.set_index('decile')['coverage'],
}, axis=1)
display(decile_coverage_comparison.style.format('{:.2%}', na_rep='N/A'))

## Inspect probability distribution

In [ ]:
def inspect_prob_dist(predictions, model_name):
    # Check probability distribution by decile
    groupby_cols = ['decile']
    grouped = predictions.groupby(groupby_cols).agg(
        orders = ('order_id', 'nunique'),
        min_prob = ('predicted_probability', 'min'),
        max_prob = ('predicted_probability', 'max'),
    ).reset_index().sort_values(groupby_cols)
    # display(grouped)

    # KDE of predicted probabilities across the monthly inference cohort.
    probabilities = predictions['predicted_probability'].dropna()
    fig, ax = plt.subplots(figsize=(10, 5))
    if probabilities.nunique() > 1:
        sns.kdeplot(x=probabilities, fill=True, clip=(0, 1), cut=0, ax=ax)
    else:
        ax.text(0.5, 0.5, 'KDE requires at least two distinct probabilities.',
                ha='center', va='center', transform=ax.transAxes)
    ax.set(
        title=f'{model_name}: monthly predicted probability distribution',
        xlabel='Predicted probability of late delivery',
        ylabel='Density',
        xlim=(0, 1),
    )
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

inspect_prob_dist(rf_ranked_predictions, "Random Forest")
inspect_prob_dist(lgb_ranked_predictions, "LightGBM")

============================================================ End of end-to-end pipeline development  ============================================================

# Time-series cross-validation and hyperparameter tuning
This section uses the same 365-day candidate window, features (`FEATURE_COLS`) and five classifiers as the June workflow. The newest 30 days form the development holdout. A 45-day gap and label-availability check are applied before that holdout.

The remaining development-training rows are evaluated using five expanding chronological CV folds. Each fold has a 30-calendar-day validation block, a 45-day gap, and training labels known at validation start. Default and tuned candidates use the same folds and holdout rows. The final June comparison refits both variants on the same full set of labels available before the run date.

The reusable code lives in `src/tuning.py`. Tuned parameters are saved to `results/feature_selection_run/classification_tuning.json`.

This section evaluates the default parameter settings on the same chronological folds and development holdout used for the tuned candidates, using the June 2018 run date.

In [ ]:
from sklearn.model_selection import ParameterGrid

sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))
from src.splits import day_blocked_time_series_folds, available_outcome_folds
from src.evaluation import classification_metrics
from src.tuning import (build_classifiers, cv_summary, tune_classifier, out_of_fold_scores,
                        best_f1_threshold, params_to_json, CLASSIFIER_SEARCH_SPACES, MODEL_FAMILY)

N_CV_FOLDS = 5
CV_VALIDATION_DAYS = 30  # Fixed calendar length for every CV validation block
# Random-search iterations per model; kept small for laptop run times
N_ITER = {'Logistic Regression': 12, 'Decision Tree': 15, 'Random Forest': 8,
          'XGBoost': 12, 'LightGBM': 12}

## Chronological split and CV folds
- Use five expanding folds with fixed 30-calendar-day validation blocks.
- Leave a 45-day gap before each block and retain only training labels known at its start.
- Keep orders from the same approval day together.

The validation blocks are counted backwards from the end of development training. For June, the buffered development-training window leaves 95 initial calendar days before the first gap. Each later fold has access to more earlier data after that data falls outside the 45-day gap. All five folds must contain both target classes for boosting.

### Display chronological CV folds

In [ ]:
# Reusing the same splits that was created in the earlier development pipeline
cv_train_df = train_df.copy()
cv_test_df = test_df.copy()

X_cv_train, y_cv_train = X_train, y_train
X_cv_test, y_cv_test = X_test, y_test


print(f'Train: {cv_train_df["order_approved_dt"].min():%Y-%m-%d} to {cv_train_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(cv_train_df):,} orders, {y_cv_train.mean():.2%} late)')
print(f'Test:  {cv_test_df["order_approved_dt"].min():%Y-%m-%d} to {cv_test_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(cv_test_df):,} orders, {y_cv_test.mean():.2%} late)')

cv_folds = day_blocked_time_series_folds(
    cv_train_df['order_approved_dt'], n_splits=N_CV_FOLDS,
    gap_days=LOOKBACK, validation_days=CV_VALIDATION_DAYS)
cv_folds = available_outcome_folds(
    cv_train_df, cv_folds, y_cv_train, labels_as_of, 'label_as_of_run')
assert len(cv_folds) == N_CV_FOLDS
assert all(y_cv_train.iloc[tr].nunique() == 2 for tr, _ in cv_folds)
print(f'CV: {len(cv_folds)} folds with a {LOOKBACK}-day gap and known training labels')
display(pd.DataFrame([{
            'fold': k, 'train_orders': len(tr), 'valid_orders': len(va),
            'valid_from': cv_train_df['order_approved_dt'].iloc[va].min().date(),
            'valid_to': cv_train_df['order_approved_dt'].iloc[va].max().date(),
            'valid_late_rate': round(y_cv_train.iloc[va].mean() * 100, 2),
                  } for k, (tr, va) in enumerate(cv_folds, 1)]
            ).set_index('fold')
      )

### Build all classifiers
- Instantiate the model pipelines

In [ ]:
# Boosting pipelines calculate scale_pos_weight from each fit's training labels.
default_models = build_classifiers(NUM_COLS, CAT_COLS, random_state=RANDOM_STATE, reference_categories=True)

## Cross-validated defaults
These are the settings used in the Model Training section above.
- **AP** is the primary metric. A random model scores about the late rate.
- **F1** uses the 0.5 threshold.

### Evaluation metrics from CV set

In [ ]:
cv_rows = []
for name, model in default_models.items():
    cv_rows.append({'model': name, 'variant': 'default', 'family': MODEL_FAMILY[name],
                    **cv_summary(model, X_cv_train, y_cv_train, cv_folds)})
cv_results = pd.DataFrame(cv_rows).set_index(['model', 'variant'])
num_cols = cv_results.select_dtypes(include='number').columns
cv_results[num_cols] = (cv_results[num_cols] * 100).round(2)
display(cv_results)

### Evaluation metrics from test set (held-out set)

In [ ]:
# Refit defaults on the full training set; score only the held-out test set.
default_test_rows = []
for name, model in default_models.items():
    fitted = clone(model).fit(X_cv_train, y_cv_train)
    prob = fitted.predict_proba(X_cv_test)[:, list(fitted.classes_).index(1)]
    default_test_rows.append({
        'model': name, 'variant': 'default', 'family': MODEL_FAMILY[name],
        **classification_metrics(y_cv_test, (prob >= PREDICTION_THRESHOLD).astype(int), prob),
    })
default_test_results = pd.DataFrame(default_test_rows).set_index(['model', 'variant'])
# Display percentages, matching the CV table; retain raw metrics for reuse.
test_display = default_test_results.copy()
test_num_cols = test_display.select_dtypes(include='number').columns
test_display[test_num_cols] = (test_display[test_num_cols] * 100).round(2)
display(test_display)

## Conduct hyperparameter tuning
1. `RandomizedSearchCV` evaluates sampled model-specific settings on the five chronological folds, maximising mean average precision (AP).
2. Out-of-fold scores are used to inspect alternative F1 thresholds; the comparison and deployment threshold remains fixed at **0.5**.
3. Default and tuned candidates are compared on the same buffered development holdout. After refitting on the full available history, their pooled June inference results choose the separate June winner, whose exact fitted pipeline is frozen for July and August. The development-holdout winner remains recorded separately.

### Random search
Each model searches a small grid (`CLASSIFIER_SEARCH_SPACES` in `src/tuning.py`) on the same five time-series folds, maximising mean average precision (AP).
- Boosting weights are recalculated from each fit's training labels; other models retain balanced class weights. Weighting uses `class_weight='balanced'`, or `scale_pos_weight` for the boosting models.
- The buffered development holdout compares default and tuned candidates after CV tuning.

In [ ]:
tuned_models, best_params = {}, {}
for name, model in default_models.items():
    n_combinations = min(N_ITER[name], len(ParameterGrid(CLASSIFIER_SEARCH_SPACES[name])))
    n_folds = len(cv_folds)
    search_fits = n_combinations * n_folds
    print(f'{name}: {n_combinations} combinations x {n_folds} folds = {search_fits} search training fits; '
          f'+ {n_folds} summary fits = {search_fits + n_folds} total fits', flush=True)
    
    tuned, params, _ = tune_classifier(model, CLASSIFIER_SEARCH_SPACES[name], X_cv_train, y_cv_train,
                                       cv_folds, n_iter=N_ITER[name], random_state=RANDOM_STATE
                                       )
    tuned_models[name], best_params[name] = tuned, params
    cv_rows.append({'model': name, 
                    'variant': 'tuned', 
                    'family': MODEL_FAMILY[name],
                    **cv_summary(tuned, X_cv_train, y_cv_train, cv_folds)
                    })
    print(f'{name}: {params_to_json(params)}')

cv_results = pd.DataFrame(cv_rows).set_index(['model', 'variant']).sort_index()
num_cols = cv_results.select_dtypes(include='number').columns
cv_results[num_cols] = (cv_results[num_cols] * 100).round(2)
display(cv_results)

cv_results_comparison = (cv_results.xs('tuned', level='variant')['cv_avg_precision_mean'] - cv_results.xs('default', level='variant')['cv_avg_precision_mean']
                         ).rename('cv_avg_precision_gain_from_tuning'
                                  ).to_frame()
display(cv_results_comparison)

### Threshold selection
For each tuned model, this picks the threshold that maximises F1 on the **out-of-fold** CV scores. Each score comes from a model that never saw that order.

In [ ]:
tuned_thresholds = {}
for name, model in tuned_models.items():
    oof = out_of_fold_scores(model, X_cv_train, y_cv_train, cv_folds)
    tuned_thresholds[name], oof_f1 = best_f1_threshold(y_cv_train, oof)
    print(f'{name}: threshold {tuned_thresholds[name]:.3f} (out-of-fold F1 {oof_f1:.3f})')

### Hold-out evaluation (newest 30 days of the window)

In [ ]:
holdout_rows = []
for variant, model_set in [('default', default_models), ('tuned', tuned_models)]:
    for name, model in model_set.items():
        fitted = clone(model).fit(X_cv_train, y_cv_train)
        prob = fitted.predict_proba(X_cv_test)[:, list(fitted.classes_).index(1)]
        cutoffs = [0.5] + ([tuned_thresholds[name]] if variant == 'tuned' else [])
        for cutoff in cutoffs:
            holdout_rows.append({'model': name, 'variant': variant, 'threshold': cutoff,
                                 'flagged_pct': (prob >= cutoff).mean(),
                                 **classification_metrics(y_cv_test, (prob >= cutoff).astype(int), prob)})

holdout_results = pd.DataFrame(holdout_rows).set_index(['model', 'variant', 'threshold'])
display(holdout_results.style.format('{:.2%}', na_rep='N/A'))


# Record the development-holdout winner
The buffered March 19–April 17 holdout compares default and tuned candidates. Its winner is recorded separately. All candidates are then refitted on the same full eligible history and compared on pooled June orders to choose the model used in the frozen-model experiment. June is consequently a model-selection cohort, not an unbiased final test of the selected winner.


In [ ]:
# Select from raw hold-out metrics, without using inference outcomes.
selection_candidates = holdout_results.reset_index().copy()
selection_candidates['deployment_threshold'] = 0.5

# Tuned models were evaluated at both 0.5 and their OOF threshold. Keep only
# the fixed 0.5 comparison threshold; hold-out outcomes do not tune it.

selection_candidates = selection_candidates.loc[np.isclose(
    selection_candidates['threshold'], selection_candidates['deployment_threshold'],
    rtol=0, atol=1e-12,
)].drop_duplicates(['model', 'variant'])

selection_candidates = selection_candidates.dropna(subset=['avg_precision']).sort_values(
    ['avg_precision', 'f1_score', 'roc_auc', 'model', 'variant'],
    ascending=[False, False, False, True, True], kind='stable',
).reset_index(drop=True)

if selection_candidates.empty:
    raise ValueError('No valid hold-out AP: check that the hold-out contains both classes.')

best_model_result = selection_candidates.iloc[0]
best_model_name = best_model_result['model']
best_model_variant = best_model_result['variant']
best_model_threshold = float(best_model_result['deployment_threshold'])
best_model_template = (tuned_models if best_model_variant == 'tuned' else default_models)[best_model_name]

best_model_description = f'{best_model_name} ({best_model_variant})'

print(f'Best on development holdout: {best_model_description}; AP '
      f'{best_model_result["avg_precision"]:.4f}, F1 {best_model_result["f1_score"]:.4f}; '
      f'prediction threshold {best_model_threshold:.4f}')

display(selection_candidates.drop(columns='deployment_threshold').style.format({
    'threshold': '{:.4f}', **{column: '{:.2%}' for column in holdout_results.columns},
}, na_rep='N/A'))


In [ ]:
# Record the holdout winner and include the holdout in its historical refit.
# The separate all-candidate June selection follows below.
# No orders from the recent buffer or inference month are fitted.

best_model_pipeline = clone(best_model_template).fit(
    train_test_label[FEATURE_COLS], train_test_label[LABEL_COL],
)

print(f'Refitted on {len(train_test_label):,} orders with labels known before {run_date}.')


In [ ]:
# Use the same fitted model for every daily cohort in this run's month.
best_inference_start_date, best_inference_end_date = get_month_start_end(run_date)

# Run daily predictions using the selected model

best_inference_predictions = run_inference(
    best_inference_start_date, best_inference_end_date, best_model_pipeline,
)

# Retrieve actual labels for each order

best_inference_predictions = attach_prediction_labels(
    best_inference_predictions, lookback=LOOKBACK,
    prediction_threshold=best_model_threshold, model_name=best_model_description,
)

# Retain model provenance alongside each saved daily score.
best_inference_predictions['model'] = best_model_name
best_inference_predictions['variant'] = best_model_variant
best_inference_predictions['model_run_date'] = run_date
best_inference_predictions['prediction_threshold'] = best_model_threshold

best_daily_prediction_summary = best_inference_predictions.groupby('order_approved_dt').agg(
        orders=('order_id', 'size'),
        flagged_orders=('predicted_label', 'sum'),
        known_labels=('actual_label', 'count'),
    ).reindex(pd.date_range(best_inference_start_date, best_inference_end_date), fill_value=0)

best_daily_prediction_summary.index.name = 'inference_date'

best_daily_prediction_summary['unknown_labels'] = (
        best_daily_prediction_summary['orders'] - best_daily_prediction_summary['known_labels']
    )

display(best_inference_predictions.head())


In [ ]:
# Compute metrics once on all pooled daily predictions, rather than averaging
# daily metrics. Unknown actual labels are reported and excluded by the helper.

best_inference_metrics = evaluate_monthly_predictions(
    best_inference_predictions, best_model_description,
)

print(f'Pooled inference evaluation: {best_inference_start_date} to {best_inference_end_date}')
display(best_inference_metrics.style.format('{:.2%}', na_rep='N/A'))

# Rank the whole monthly cohort together, using the same decile definitions
# and unknown-label handling as the earlier inference backtest.

best_inference_coverage, best_ranked_predictions = display_decile_coverage(
    best_inference_predictions, best_model_description,
)


## June model selection: default-versus-tuned comparison
Each candidate is fitted on the same full 365-day eligible history known before June 2. We pool its daily June predictions and select the highest AP, breaking ties by F1 and ROC-AUC. The winner's already fitted pipeline is kept unchanged for July and August: no refitting, retuning or threshold adjustment.

This is a retrospective frozen-model experiment. June 30 outcomes at the prescribed 45-day horizon are assessed through August 14 (available August 15 under the exclusive-date helper). A winner selected from the complete June cohort could not have been chosen on July 1. July and early August are therefore retrospective later-cohort stress tests, not a prospective deployment simulation.


In [ ]:
# Refit and compare every default/tuned candidate on the same June inference cohort.
june_candidate_rows = []
june_candidate_predictions = {}
june_fitted_candidates = {}
reference_order_ids = None
reference_actual_labels = None

for variant, model_set in [('default', default_models), ('tuned', tuned_models)]:
    for name, model in model_set.items():
        fitted = clone(model).fit(
            train_test_label[FEATURE_COLS], train_test_label[LABEL_COL],
        )
        predictions = run_inference(
            best_inference_start_date, best_inference_end_date, fitted,
        )
        predictions = attach_prediction_labels(
            predictions,
            lookback=LOOKBACK,
            prediction_threshold=PREDICTION_THRESHOLD,
            model_name=f'{name} ({variant})',
        )

        ordered_ids = tuple(predictions['order_id'])
        actual_labels = predictions.set_index('order_id')['actual_label'].sort_index()
        if reference_order_ids is None:
            reference_order_ids = ordered_ids
            reference_actual_labels = actual_labels
        else:
            assert ordered_ids == reference_order_ids, 'Candidates were scored on different June orders.'
            assert actual_labels.equals(reference_actual_labels), 'Candidates received different June labels.'

        metrics = evaluate_monthly_predictions(predictions, f'{name} ({variant})').iloc[0]
        known = predictions.dropna(subset=['actual_label'])
        late_count = int(known['actual_label'].astype(int).sum())
        ranked = predictions.sort_values(
            ['predicted_probability', 'order_id'],
            ascending=[False, True], kind='stable',
        )
        top_decile_size = max(1, int(np.ceil(len(ranked) * 0.10))) if len(ranked) else 0
        top_decile_lates = int(
            ranked.head(top_decile_size)['actual_label'].fillna(0).astype(int).sum()
        )

        june_candidate_rows.append({
            'model': name,
            'family': MODEL_FAMILY[name],
            'variant': variant,
            'scored_orders': len(predictions),
            'evaluated_orders': len(known),
            'known_late_orders': late_count,
            'AP': metrics['avg_precision'],
            'ROC-AUC': metrics['roc_auc'],
            'precision_at_0_5': metrics['precision'],
            'recall_at_0_5': metrics['recall'],
            'F1_at_0_5': metrics['f1_score'],
            'top_decile_late_coverage': top_decile_lates / late_count if late_count else np.nan,
            'top_decile_late_orders': top_decile_lates,
        })
        june_candidate_predictions[(name, variant)] = predictions
        june_fitted_candidates[(name, variant)] = fitted

june_candidate_results = pd.DataFrame(june_candidate_rows).set_index(['model', 'variant']).sort_index()

print(f'June cohort: {best_inference_start_date} to {best_inference_end_date}; '
      f'{len(reference_order_ids):,} scored orders, '
      f'{june_candidate_results["evaluated_orders"].iloc[0]:,} with known outcomes.')

display(june_candidate_results.style.format({
    'AP': '{:.2%}', 'ROC-AUC': '{:.2%}',
    'precision_at_0_5': '{:.2%}', 'recall_at_0_5': '{:.2%}',
    'F1_at_0_5': '{:.2%}', 'top_decile_late_coverage': '{:.2%}',
}, na_rep='N/A'))

# Positive gains mean the tuned candidate improved that metric versus its own default.
june_tuning_gain_rows = []

for name in default_models:
    default = june_candidate_results.loc[(name, 'default')]
    tuned = june_candidate_results.loc[(name, 'tuned')]
    june_tuning_gain_rows.append({
        'model': name,
        'family': MODEL_FAMILY[name],
        'default_AP': default['AP'],
        'tuned_AP': tuned['AP'],
        'AP_gain_pp': (tuned['AP'] - default['AP']) * 100,
        'ROC_AUC_gain_pp': (tuned['ROC-AUC'] - default['ROC-AUC']) * 100,
        'precision_gain_pp': (tuned['precision_at_0_5'] - default['precision_at_0_5']) * 100,
        'recall_gain_pp': (tuned['recall_at_0_5'] - default['recall_at_0_5']) * 100,
        'F1_gain_pp': (tuned['F1_at_0_5'] - default['F1_at_0_5']) * 100,
        'top_decile_coverage_gain_pp': (
            tuned['top_decile_late_coverage'] - default['top_decile_late_coverage']
        ) * 100,
    })

june_tuning_gains = pd.DataFrame(june_tuning_gain_rows).set_index(['model', 'family'])
print('Tuned-minus-default change within each model family (AP gain is in percentage points).')
display(june_tuning_gains.style.format('{:+.2f}', na_rep='N/A'))


## June-selected frozen model and later-cohort performance
The June winner is chosen by AP from all default and tuned candidates. July and August reuse the identical fitted model and preprocessing; their outcomes play no role in this choice. Permutation importance on June describes the selection cohort and is not independent validation.


In [ ]:
from pathlib import Path
from sklearn.inspection import permutation_importance
import hashlib
import pickle

june_ranked = june_candidate_results.reset_index().sort_values(
    ['AP', 'F1_at_0_5', 'ROC-AUC', 'model', 'variant'],
    ascending=[False, False, False, True, True], kind='stable')

deployment_result = june_ranked.iloc[0]
deployment_model_name = deployment_result['model']
deployment_variant = deployment_result['variant']
deployment_pipeline = june_fitted_candidates[(deployment_model_name, deployment_variant)]
deployment_description = f'{deployment_model_name} ({deployment_variant})'
frozen_fingerprint = hashlib.sha256(pickle.dumps(deployment_pipeline)).hexdigest()
print(f'June-selected model: {deployment_description}; June AP {deployment_result["AP"]:.2%}')
print(f'Development-holdout winner: {best_model_description} (separate selection stage)')

frozen_predictions = {}
frozen_rows = []

for month in ['2018-06', '2018-07', '2018-08']:
    start = pd.Timestamp(month + '-01')
    end = start + pd.offsets.MonthEnd(0)
    if month == '2018-06':
        predictions = june_candidate_predictions[(deployment_model_name, deployment_variant)].copy()
    else:
        predictions = attach_prediction_labels(
            run_inference(str(start.date()), str(end.date()), deployment_pipeline),
            model_name=deployment_description)
    assert hashlib.sha256(pickle.dumps(deployment_pipeline)).hexdigest() == frozen_fingerprint
    metrics = evaluate_monthly_predictions(predictions, deployment_description).iloc[0]
    known = predictions.dropna(subset=['actual_label'])
    late_count = int(known['actual_label'].sum())
    ranked = predictions.sort_values(['predicted_probability', 'order_id'], ascending=[False, True])
    top_count = int(np.ceil(len(ranked) * .1))
    captured = int(ranked.head(top_count)['actual_label'].fillna(0).sum())
    frozen_rows.append({
        'month': month, 'model': deployment_model_name, 'variant': deployment_variant,
        'scored_orders': len(predictions), 'evaluated_orders': len(known),
        'late_orders': late_count, 'late_rate': late_count / len(known),
        'AP': metrics['avg_precision'], 'ROC-AUC': metrics['roc_auc'],
        'precision': metrics['precision'], 'recall': metrics['recall'], 'F1': metrics['f1_score'],
        'top_decile_capture': captured / late_count if late_count else np.nan,
        'top_decile_lift': (captured / late_count) / (top_count / len(ranked)) if late_count else np.nan,
        'fit_run_date': run_date, 'refitted': False,
    })
    frozen_predictions[month] = predictions.assign(month=month)

frozen_results = pd.DataFrame(frozen_rows)
display(frozen_results)

# Explain the June-selected pipeline on the June model-selection cohort.
importance_ids = frozen_predictions['2018-06'].dropna(subset=['actual_label'])['order_id']
importance_features = final_df.set_index('order_id').loc[importance_ids, FEATURE_COLS]
importance_labels = frozen_predictions['2018-06'].set_index('order_id').loc[importance_ids, 'actual_label'].astype(int)

importance = permutation_importance(
    deployment_pipeline, importance_features, importance_labels,
    scoring='average_precision', n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)

importance_table = pd.DataFrame({
    'feature': FEATURE_COLS, 'AP_decrease': importance.importances_mean,
    'AP_decrease_sd': importance.importances_std,
}).sort_values('AP_decrease', ascending=False)

display(importance_table.head(10))
fig, ax = plt.subplots(figsize=(8, 4))
top_features = importance_table.head(10).iloc[::-1]
ax.barh(top_features['feature'], top_features['AP_decrease'], xerr=top_features['AP_decrease_sd'])
ax.set_xlabel('Decrease in average precision after permutation')
ax.set_ylabel('')
fig.tight_layout()
Path('results/feature_selection_run/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('results/feature_selection_run/figures/classification_june_selected_importance.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown
from src.report_tables import export_report_tables

june_output_dir = Path('results/feature_selection_run/report_tables')
june_output_dir.mkdir(parents=True, exist_ok=True)

# Format the all-candidate comparison on the June selection cohort.
june_table = june_candidate_results.reset_index().copy()

june_table['Selected'] = (
    june_table['model'].eq(deployment_model_name)
    & june_table['variant'].eq(deployment_variant)
).map({True: 'Yes', False: ''})

june_table = june_table.rename(columns={
    'model': 'Model', 'variant': 'Configuration',
    'evaluated_orders': 'Evaluated orders', 'known_late_orders': 'Late orders',
    'AP': 'AP (%)', 'ROC-AUC': 'ROC-AUC',
    'precision_at_0_5': 'Precision (%)', 'recall_at_0_5': 'Recall (%)',
    'F1_at_0_5': 'F1 (%)', 'top_decile_late_coverage': 'Top-decile capture (%)',
})

june_table['AP (%)'] = (100 * june_table['AP (%)']).map(lambda value: f'{value:.2f}')
june_table['ROC-AUC'] = june_table['ROC-AUC'].map(lambda value: f'{value:.3f}')

for column in ['Precision (%)', 'Recall (%)', 'F1 (%)', 'Top-decile capture (%)']:
    june_table[column] = (100 * june_table[column]).map(lambda value: f'{value:.2f}')

june_table = june_table[['Model', 'Configuration', 'Selected', 'AP (%)', 'ROC-AUC',
                             'Precision (%)', 'Recall (%)', 'F1 (%)',
                             'Top-decile capture (%)']]

# Show the tuning change in percentage points within each model family.

june_gain_table = june_tuning_gains.reset_index().rename(columns={
    'model': 'Model', 'AP_gain_pp': 'ΔAP (pp)', 'ROC_AUC_gain_pp': 'ΔROC-AUC (pp)',
})[['Model', 'ΔAP (pp)', 'ΔROC-AUC (pp)']]

for column in ['ΔAP (pp)', 'ΔROC-AUC (pp)']:
    june_gain_table[column] = june_gain_table[column].map(lambda value: f'{value:+.2f}')

# Save auditable full-precision rows and the June cohort predictions.
june_candidate_results.reset_index().to_csv(
    june_output_dir / 'classification_june_candidate_metrics.csv', index=False)
june_tuning_gains.reset_index().to_csv(
    june_output_dir / 'classification_june_tuning_gains.csv', index=False)
frozen_predictions['2018-06'].to_csv(
    june_output_dir / 'classification_june_selected_predictions.csv', index=False)

selected_holdout_ap = float(best_model_result['avg_precision'])
selected_holdout_roc_auc = float(best_model_result['roc_auc'])

selection_text = (
    f'June holdout selection: **{best_model_name} ({best_model_variant})** '
    f'(holdout AP {selected_holdout_ap:.2%}; ROC-AUC {selected_holdout_roc_auc:.3f}; '
    f'threshold {best_model_threshold:.2f}).'
)

print(selection_text)

june_evaluation_note = (
    f'June inference orders scored: {len(best_inference_predictions):,}; '
    f'known outcomes: {best_inference_predictions["actual_label"].notna().sum():,}; '
    f'late outcomes: {best_inference_predictions["actual_label"].fillna(0).sum():,.0f}.'
)

june_report_tables = [
    {'title': 'June classification performance', 'frame': june_table,
     'note': 'AP, precision, recall, F1 and top-decile capture are percentages. ROC-AUC is on a 0–1 scale. Threshold metrics use 0.5. “Selected” marks the model chosen retrospectively using June AP. June is a selection cohort; July and August assess the unchanged fitted model. ' + june_evaluation_note,
     'prose': selection_text},
    {'title': 'June tuned-minus-default change (percentage points)', 'frame': june_gain_table,
     'note': 'Change = tuned minus default on the same June evaluation orders; positive values indicate improvement.',
     'prose': 'This table summarizes the June inference gain associated with tuning within each model family.'},
]

for table in june_report_tables:
    display(Markdown('### ' + table['title']))
    display(table['frame'])
    display(Markdown('**Note.** ' + table['note'] + '\n\n' + table['prose']))

june_report_paths = export_report_tables(
    june_report_tables, june_output_dir, 'classification_june_performance')
print(june_report_paths)


# Save the June tuning audit with the exact split definitions and selected parameters.
fold_audit = []

for fold_number, (train_indices, validation_indices) in enumerate(cv_folds, start=1):
    validation_dates = cv_train_df.iloc[validation_indices]['order_approved_dt']
    fold_audit.append({
        'fold': fold_number,
        'train_orders': int(len(train_indices)),
        'validation_orders': int(len(validation_indices)),
        'validation_start': validation_dates.min().strftime('%Y-%m-%d'),
        'validation_end': validation_dates.max().strftime('%Y-%m-%d'),
    })

model_audit = {}

for name in default_models:
    model_audit[name] = {
        'family': MODEL_FAMILY[name],
        'best_params': params_to_json(best_params[name]),
        'f1_threshold_oof': float(tuned_thresholds[name]),
        'cv_avg_precision_mean_pct': {
            variant: float(cv_results.loc[(name, variant), 'cv_avg_precision_mean'])
            for variant in ['default', 'tuned']
        },
    }

tuning_audit = {
    'run_date': run_date,
    'inference_month': pd.Timestamp(best_inference_start_date).strftime('%Y-%m'),
    'history_window': {
        'start': str(train_start_date), 'end': str(test_end_date),
        'calendar_days': int(PXD), 'inference_buffer_days': int(LOOKBACK),
    },
    'development_split': {
        'cv_training_start': cv_train_df['order_approved_dt'].min().strftime('%Y-%m-%d'),
        'cv_training_end': cv_train_df['order_approved_dt'].max().strftime('%Y-%m-%d'),
        'cv_training_orders': int(len(cv_train_df)),
        'holdout_start': cv_test_df['order_approved_dt'].min().strftime('%Y-%m-%d'),
        'holdout_end': cv_test_df['order_approved_dt'].max().strftime('%Y-%m-%d'),
        'holdout_orders': int(len(cv_test_df)),
        'gap_days': int(LOOKBACK),
        'historical_orders': int(len(train_test_label_audit)),
        'known_labels_at_run_date': int(len(train_test_label)),
        'unknown_labels_at_run_date': int(len(train_test_label_audit) - len(train_test_label)),
    },
    'cv': {
        'scheme': 'day-blocked expanding window',
        'validation_days': int(CV_VALIDATION_DAYS),
        'gap_days': int(LOOKBACK),
        'training_outcomes_checked_at_validation_start': True,
        'folds': fold_audit,
        'scoring': 'average_precision',
        'cv_average_precision_mean_pct_is_rounded': True,
    },
    'search': {
        'method': 'RandomizedSearchCV', 'random_state': int(RANDOM_STATE),
        'iterations': {name: int(min(N_ITER[name], len(ParameterGrid(CLASSIFIER_SEARCH_SPACES[name]))) )
                       for name in default_models},
    },
    'models': model_audit,
    'development_holdout_selection': {
        'metric': 'average_precision', 'tie_breakers': ['F1', 'ROC-AUC'],
        'threshold': float(best_model_threshold),
        'selected_model': best_model_name, 'selected_variant': best_model_variant,
        'holdout_average_precision': float(best_model_result['avg_precision']),
        'holdout_roc_auc': float(best_model_result['roc_auc']),
    },
    'june_model_selection': {'selected_model': deployment_model_name, 'selected_variant': deployment_variant,
        'AP': float(deployment_result['AP']), 'criterion': 'June pooled AP, then F1, then ROC-AUC',
        'evaluation_horizon_end': '2018-08-14', 'full_cohort_decision_available_from': '2018-08-15',
        'later_months_refitted': False,
        'interpretation': 'Retrospective frozen-model stress test; July selection was not available prospectively'},
    'final_refit_orders': int(len(train_test_label)),
    'inference_evaluation': {
        'scored_orders': int(len(best_inference_predictions)),
        'evaluated_orders': int(best_inference_predictions['actual_label'].notna().sum()),
        'late_orders': int(best_inference_predictions['actual_label'].fillna(0).sum()),
    },
}

with (Path('results/feature_selection_run') / 'classification_tuning.json').open('w') as audit_file:
    json.dump(tuning_audit, audit_file, indent=2)
    audit_file.write('\n')

print('Saved June tuning audit to results/feature_selection_run/classification_tuning.json')

# Export compact later-cohort results alongside the June candidate comparison.
frozen_results.to_csv(june_output_dir / 'classification_frozen_monthly_metrics.csv', index=False)
pd.concat(frozen_predictions.values(), ignore_index=True).to_csv(
    june_output_dir / 'classification_frozen_predictions.csv', index=False)
importance_table.to_csv(june_output_dir / 'classification_june_selected_importance.csv', index=False)
selection_candidates.to_csv(june_output_dir / 'classification_june_holdout_metrics.csv', index=False)
cv_results.reset_index().to_csv(june_output_dir / 'classification_june_cv_metrics.csv', index=False)
drift_table = frozen_results[['month', 'evaluated_orders', 'late_rate', 'AP', 'ROC-AUC',
                              'top_decile_capture', 'top_decile_lift']].copy()

for column in ['late_rate', 'AP', 'top_decile_capture']:
    drift_table[column] = drift_table[column].map(lambda x: f'{100*x:.2f}')

for column in ['ROC-AUC', 'top_decile_lift']:
    drift_table[column] = drift_table[column].map(lambda x: f'{x:.3f}')

drift_table.columns = ['Month', 'Orders', 'Late (%)', 'AP (%)', 'ROC-AUC', 'Top-10% capture (%)', 'Lift']

june_report_tables.append({
    'title': 'June-selected model: frozen July and August predictions', 'frame': drift_table,
    'note': f'{deployment_description}; fitted once using June historical data. June selects the model; later months never refit or retune it. Complete June horizon outcomes are available August 15, so July and early August are retrospective stress tests. Unknown outcomes are excluded; the latest recorded delivery is an unverified observation-end proxy.',
    'prose': 'Monthly differences show performance stability for the same fitted model, not changes caused by monthly retraining.',
})

export_report_tables(june_report_tables, june_output_dir, 'classification_june_performance')
display(drift_table)


## June-selected classifier: capture and lift by decile

Each decile contains approximately 10% of scored June orders, ranked from highest to lowest risk. Capture and lift describe individual deciles; cumulative capture shows the late orders reached as intervention capacity increases. This ranking does not use the 0.5 classification threshold.

In [ ]:
from src.evaluation import risk_decile_coverage

# Use the exact June predictions that selected the frozen classification model.
june_deciles = risk_decile_coverage(frozen_predictions['2018-06'])
june_deciles.to_csv(
    june_output_dir / 'classification_june_decile_coverage.csv', index=False,
)

# Keep the report compact; preserve full-precision counts and values in the CSV.
decile_table = june_deciles[[
    'decile', 'scored_orders', 'late_orders', 'capture_pct', 'lift',
    'cumulative_capture_pct',
]].copy()
decile_table.columns = [
    'Risk decile', 'Orders', 'Late orders', 'Capture (%)', 'Lift',
    'Cumulative capture (%)',
]

for column in ['Capture (%)', 'Lift', 'Cumulative capture (%)']:
    decile_table[column] = decile_table[column].map(lambda value: f'{value:.2f}')

unknown_count = int((june_deciles['scored_orders'] - june_deciles['known_orders']).sum())
decile_report = {
    'title': f'June risk deciles: {deployment_description}',
    'frame': decile_table,
    'note': (
        'Decile 1 has the highest risk. Capture is the share of all known late '
        'orders within each decile; cumulative capture includes all higher-risk '
        'deciles. Lift is capture share divided by scored-order share, matching '
        'the existing top-10% metric; 1 represents random selection. '
        f'{unknown_count} unknown outcomes remain ranked but do not count as late. '
        'June informed model selection; these results are descriptive.'
    ),
    'prose': (
        'Individual-decile lift shows where late orders concentrate; cumulative '
        'capture shows how many late orders could be reached at increasing contact capacity.'
    ),
}

display(decile_table)
display(Markdown('**Note.** ' + decile_report['note']))
export_report_tables([decile_report], june_output_dir, 'classification_june_deciles')

# Include the same table at the end of the combined classification report.
june_report_tables = [
    table for table in june_report_tables
    if not table['title'].startswith('June risk deciles:')
]
june_report_tables.append(decile_report)
export_report_tables(june_report_tables, june_output_dir, 'classification_june_performance')
